In [ ]:
from pathlib import Path
import os
import random
import logging
import pandas as pd
import requests
from citipy import citipy

weather_api_key = os.environ['OPENWEATHER_API_KEY']
rng = random.Random(42)
coordinates = [(rng.uniform(-90, 90), rng.uniform(-180, 180)) for _ in range(2000)]
cities = sorted({citipy.nearest_city(latitude, longitude).city_name for latitude, longitude in coordinates})
print(f'City count: {len(cities)}')


In [ ]:
import time

city_data = []
next_request = 0.0
with requests.Session() as client:
    for city in cities:
        time.sleep(max(0.0, next_request - time.monotonic()))
        next_request = time.monotonic() + 1.1
        response = client.get('https://api.openweathermap.org/data/2.5/weather', params={'q': city, 'appid': weather_api_key, 'units': 'metric'}, timeout=30)
        if response.status_code == 404:
            logging.info('No weather record for %s', city)
            continue
        if not response.ok:
            raise RuntimeError(f'API request failed with status {response.status_code}.')
        weather = response.json()
        city_data.append({
            'City': city,
            'Country': weather['sys']['country'],
            'Latitude': weather['coord']['lat'],
            'Longitude': weather['coord']['lon'],
            'Max Temperature': weather['main']['temp_max'],
            'Humidity': weather['main']['humidity'],
            'Cloudiness': weather['clouds']['all'],
            'Wind Speed': weather['wind']['speed'],
            'Weather Description': weather['weather'][0]['description']
        })
columns = ['City', 'Country', 'Latitude', 'Longitude', 'Max Temperature', 'Humidity', 'Cloudiness', 'Wind Speed', 'Weather Description']
city_data_df = pd.DataFrame(city_data, columns=columns)
print(city_data_df.head())


In [ ]:
output_dir = Path('Weather_Database') if Path('Weather_Database').is_dir() else Path('.')
output_file = output_dir / 'WeatherPy_database.csv'
city_data_df.to_csv(output_file, index=False)
print(output_file)
